# Bloque A: Preparación, EDA y Evaluación de Normalizadores

**Objetivo:** Cargar el dataset Fashion MNIST, realizar un Análisis Exploratorio de Datos (EDA) exhaustivo y evaluar sistemáticamente diferentes estrategias de normalización para seleccionar las más adecuadas para las fases posteriores de reducción y clasificación.

**Autor:** Jules
**SEMILLA:** 42

**Nota Importante:** Este notebook no asume una única estrategia de normalización (como la simple división por 255). En su lugar, se evaluarán múltiples alternativas de forma empírica para tomar una decisión informada.

## 1. Configuración del Entorno

Importamos las librerías necesarias, configuramos las semillas para reproducibilidad y definimos constantes globales.

In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, Normalizer as SklearnNormalizer
from sklearn.decomposition import PCA
from skimage.exposure import equalize_adapthist
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

ruta_modulo = os.path.abspath(os.path.join('..'))
if ruta_modulo not in sys.path:
    sys.path.append(ruta_modulo)

from src.data import loader as cargador
from src.analysis import eda
from src.utils import helpers as ayudantes

SEMILLA = 42
np.random.seed(SEMILLA)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')

NOMBRES_CLASES = [
    'Camiseta', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
    'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín'
]

print("Entorno configurado.")

### Verificación de Dependencias

In [ ]:
with open('../requirements.txt', 'r') as f:
    print(f.read())

## 2. Carga de Datos

In [ ]:
rutas_crudos, ruta_baseline, metadatos = cargador.ejecutar_pipeline_carga(
    dir_crudos="../data/raw/",
    dir_procesados="../data/processed/",
    sobrescribir=False
)

print("\n--- Resumen de Carga ---")
print(f"Metadatos guardados en: {metadatos['archivos_crudos']['metadata']}")
print(f"Datos crudos guardados en: {os.path.dirname(rutas_crudos['X_train'])}")
print(f"Datos baseline (minmax) guardados en: {ruta_baseline}")

### Carga de Datos Crudos en Memoria y Verificaciones

In [ ]:
x_train_crudo = np.load(rutas_crudos['X_train'])
y_train_crudo = np.load(rutas_crudos['y_train'])
x_test_crudo = np.load(rutas_crudos['X_test'])
y_test_crudo = np.load(rutas_crudos['y_test'])

print("Formas de los datos crudos:")
print(f"X_train: {x_train_crudo.shape}, y_train: {y_train_crudo.shape}")
print(f"X_test: {x_test_crudo.shape}, y_test: {y_test_crudo.shape}")

assert x_train_crudo.shape == (60000, 28, 28)
assert x_test_crudo.shape == (10000, 28, 28)
assert y_train_crudo.min() >= 0 and y_train_crudo.max() <= 9
assert x_train_crudo.min() >= 0 and x_train_crudo.max() <= 255
assert x_train_crudo.dtype == 'uint8'

print("\nChecks de integridad pasados exitosamente.")

## 3. Análisis Exploratorio de Datos (EDA)

### 3.1. Estadísticas de Clases

In [ ]:
conteo_clases = pd.Series(y_train_crudo).value_counts().sort_index()
conteo_clases.index = NOMBRES_CLASES
conteo_clases.name = 'numero_de_muestras'

os.makedirs('../outputs/tables/', exist_ok=True)
conteo_clases.to_csv('../outputs/tables/conteo_clases.csv')

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=conteo_clases.index, y=conteo_clases.values, ax=ax)
ax.set_title('Distribución de Clases en el Conjunto de Entrenamiento')
ax.set_ylabel('Número de Muestras')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()

ayudantes.guardar_figura(fig, '../outputs/figures/EDA_distribucion_clases')
plt.show()

### 3.2. Visualización de Muestras

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
axes = axes.flatten()
for i in range(len(axes)):
    idx = np.random.randint(0, x_train_crudo.shape[0])
    axes[i].imshow(x_train_crudo[idx], cmap='gray')
    axes[i].set_title(f"{NOMBRES_CLASES[y_train_crudo[idx]]}", fontsize=10)
    axes[i].axis('off')
fig.suptitle('Muestras Aleatorias del Dataset', fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.96])

ayudantes.guardar_figura(fig, '../outputs/figures/EDA_grid_muestras')
plt.show()

### 3.3. Imágenes Promedio por Clase

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()
for i, ax in enumerate(axes):
    imagen_promedio = x_train_crudo[y_train_crudo == i].mean(axis=0)
    ax.imshow(imagen_promedio, cmap='gray')
    ax.set_title(f'Promedio: {NOMBRES_CLASES[i]}')
    ax.axis('off')
fig.suptitle('Imagen Promedio por Clase', fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.95])

ayudantes.guardar_figura(fig, '../outputs/figures/EDA_imagenes_promedio')
plt.show()

### 3.4. Análisis Cuantitativo de la Dimensionalidad

#### PCA Exploratorio: Varianza Acumulada

In [ ]:
x_train_plano = x_train_crudo.reshape(x_train_crudo.shape[0], -1)

serie_var_acum, umbrales_pca = eda.calcular_varianza_acumulada_pca(x_train_plano, max_componentes=300)

fig, ax = plt.subplots(figsize=(10, 6))
serie_var_acum.plot(ax=ax)
ax.axhline(y=0.90, color='r', linestyle='--', label='90% Varianza')
ax.axhline(y=0.95, color='g', linestyle='--', label='95% Varianza')
ax.axvline(x=umbrales_pca['n_componentes_90_varianza'], color='r', linestyle=':', alpha=0.8)
ax.axvline(x=umbrales_pca['n_componentes_95_varianza'], color='g', linestyle=':', alpha=0.8)
ax.set_title('Varianza Acumulada Explicada por Componentes Principales')
ax.set_xlabel('Número de Componentes Principales')
ax.set_ylabel('Varianza Acumulada Explicada')
ax.legend()
ax.set_ylim(0.5, 1.01)
ax.set_xlim(0, len(serie_var_acum))

ayudantes.guardar_figura(fig, '../outputs/figures/EDA_pca_varianza_acumulada')
plt.show()

#### Información Mutua (Supervisada)

In [ ]:
serie_mi = eda.calcular_informacion_mutua_componentes(x_train_plano, y_train_crudo, n_componentes=100)
serie_mi.to_csv('../outputs/tables/informacion_mutua_componentes.csv')

fig, ax = plt.subplots(figsize=(12, 7))
serie_mi.head(50).plot(kind='bar', ax=ax)
ax.set_title('Información Mutua entre los Primeros 100 Componentes de PCA y la Clase')
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Información Mutua')
plt.xticks(rotation=90)
fig.tight_layout()

ayudantes.guardar_figura(fig, '../outputs/figures/EDA_informacion_mutua_componentes')
plt.show()

#### Estimación de la Dimensión Intrínseca

In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.8, random_state=SEMILLA)
train_idx, _ = next(sss.split(x_train_plano, y_train_crudo))
x_submuestra = x_train_plano[train_idx]

estimacion_dim_twonn = eda.estimar_dimension_intrinseca_twonn(x_submuestra, k=10)

estimaciones_dim = {
    'TwoNN': estimacion_dim_twonn,
    'PCA_95_var': umbrales_pca['n_componentes_95_varianza']
}

pd.Series(estimaciones_dim).to_csv('../outputs/tables/estimaciones_dim_intrinseca.csv')
print(f"\nEstimaciones guardadas.")

## 4. Evaluación de Estrategias de Normalización

### 4.1. Definición de Clases y Funciones de Normalización

Para hacer el notebook autocontenido y evitar problemas de importación, definimos las clases de normalización directamente aquí.

In [ ]:
class EstandarizadorPorImagen(BaseEstimator, TransformerMixin):
    """Normaliza cada imagen (fila) individualmente para tener media 0 y desviación estándar 1."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if X.ndim > 2:
            X_plano = X.reshape(X.shape[0], -1)
        else:
            X_plano = X.copy()

        media = np.mean(X_plano, axis=1, keepdims=True)
        std = np.std(X_plano, axis=1, keepdims=True)
        std[std == 0] = 1.0
        X_transformado = (X_plano - media) / std
        if X.ndim > 2:
            return X_transformado.reshape(X.shape)
        return X_transformado

class TransformadorBlanqueoPCA(BaseEstimator, TransformerMixin):
    """Aplica PCA y luego blanquea los datos (whiten=True)."""
    def __init__(self, n_componentes=None, semilla_aleatoria=42):
        self.n_componentes = n_componentes
        self.semilla_aleatoria = semilla_aleatoria
        self.pca = None

    def fit(self, X, y=None):
        if X.ndim > 2:
            X = X.reshape(X.shape[0], -1)
        self.pca = PCA(n_components=self.n_componentes, whiten=True, random_state=self.semilla_aleatoria)
        self.pca.fit(X)
        return self

    def transform(self, X):
        if not self.pca:
            raise RuntimeError("El transformador PCA no ha sido ajustado. Llama a 'fit' primero.")
        if X.ndim > 2:
            X = X.reshape(X.shape[0], -1)
        return self.pca.transform(X)

class TransformadorCLAHE(BaseEstimator, TransformerMixin):
    """Aplica Contrast Limited Adaptive Histogram Equalization (CLAHE)."""
    def __init__(self, limite_clip=0.03, tamano_kernel=None):
        self.limite_clip = limite_clip
        self.tamano_kernel = tamano_kernel

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if X.ndim != 3:
            raise ValueError(f"CLAHE espera datos 3D, pero recibió {X.ndim}D.")
        X_norm = X.copy()
        if X.max() > 1.0:
            X_norm = X_norm / 255.0
        X_clahe = np.array([
            equalize_adapthist(img, kernel_size=self.tamano_kernel, clip_limit=self.limite_clip)
            for img in X_norm
        ])
        return X_clahe.reshape(X_clahe.shape[0], -1)

def obtener_normalizador(nombre):
    """Factory que devuelve un objeto normalizador basado en su nombre."""
    normalizadores = {
        'minmax': MinMaxScaler(),
        'standard': StandardScaler(),
        'robust': RobustScaler(),
        'l2': SklearnNormalizer(norm='l2'),
        'por_imagen': EstandarizadorPorImagen(),
        'blanqueo_pca': TransformadorBlanqueoPCA(semilla_aleatoria=SEMILLA),
        'clahe': TransformadorCLAHE()
    }
    if nombre.lower() in normalizadores:
        return normalizadores[nombre.lower()]
    else:
        raise ValueError(f"Normalizador '{nombre}' no reconocido.")

print("Clases y funciones de normalización definidas localmente.")

### 4.2. Bucle de Evaluación

In [ ]:
NORMALIZADORES_A_EVALUAR = ['minmax', 'standard', 'robust', 'l2', 'por_imagen', 'blanqueo_pca', 'clahe']

sss_norm = StratifiedShuffleSplit(n_splits=1, train_size=10000, test_size=2000, random_state=SEMILLA)
idx_train_norm, idx_test_norm = next(sss_norm.split(x_train_crudo, y_train_crudo))

x_train_sub, y_train_sub = x_train_crudo[idx_train_norm], y_train_crudo[idx_train_norm]
x_test_sub, y_test_sub = x_train_crudo[idx_test_norm], y_train_crudo[idx_test_norm]

print(f"Submuestra para evaluación: Train={x_train_sub.shape}, Test={x_test_sub.shape}")

resultados = []

for nombre in NORMALIZADORES_A_EVALUAR:
    print(f"\n--- Evaluando: {nombre} ---")
    
    normalizador = obtener_normalizador(nombre)
    
    pipeline = Pipeline([
        ('normalizador', normalizador),
        ('pca', PCA(n_components=30, random_state=SEMILLA)),
        ('clasificador', LogisticRegression(max_iter=1000, random_state=SEMILLA))
    ])
    
    x_train_fit = x_train_sub
    x_test_fit = x_test_sub
    if nombre not in ['clahe']:
        x_train_fit = x_train_sub.reshape(len(x_train_sub), -1)
        x_test_fit = x_test_sub.reshape(len(x_test_sub), -1)
        
    tiempo_inicio = time.time()
    pipeline.fit(x_train_fit, y_train_sub)
    y_pred = pipeline.predict(x_test_fit)
    tiempo_fin = time.time()
    
    f1 = f1_score(y_test_sub, y_pred, average='macro')
    acc = accuracy_score(y_test_sub, y_pred)
    duracion = tiempo_fin - tiempo_inicio
    
    resultados.append({
        'normalizador': nombre,
        'f1_macro': f1,
        'accuracy': acc,
        'tiempo_s': duracion
    })
    print(f"Resultados: F1 Macro={f1:.4f}, Accuracy={acc:.4f}, Tiempo={duracion:.2f}s")

df_resultados = pd.DataFrame(resultados).sort_values('f1_macro', ascending=False).set_index('normalizador')
df_resultados.to_csv('../outputs/tables/comparacion_normalizadores.csv')

print("\n--- Tabla Comparativa de Normalizadores ---")
display(df_resultados)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

df_resultados['f1_macro'].plot(kind='bar', ax=ax[0], color=sns.color_palette('viridis', len(df_resultados)))
ax[0].set_title('Comparación de F1 Score (Macro) por Normalizador')
ax[0].set_ylabel('F1 Score (Macro)')
ax[0].tick_params(axis='x', rotation=45)

df_resultados['tiempo_s'].plot(kind='bar', ax=ax[1], color=sns.color_palette('plasma', len(df_resultados)))
ax[1].set_title('Tiempo de Ejecución del Pipeline por Normalizador')
ax[1].set_ylabel('Tiempo (s)')
ax[1].tick_params(axis='x', rotation=45)

fig.tight_layout()
ayudantes.guardar_figura(fig, '../outputs/figures/comparacion_normalizadores')
plt.show()

## 5. Decisión y Generación de Datasets Procesados

In [ ]:
normalizadores_elegidos = df_resultados.head(2).index.tolist()
if 'minmax' not in normalizadores_elegidos:
    normalizadores_elegidos[-1] = 'minmax'

print(f"Normalizadores seleccionados: {normalizadores_elegidos}")

for nombre in normalizadores_elegidos:
    print(f"\nGenerando dataset para '{nombre}'...")
    normalizador = obtener_normalizador(nombre)
    
    x_train_a_norm = x_train_crudo
    x_test_a_norm = x_test_crudo
    if nombre not in ['clahe']:
        x_train_a_norm = x_train_crudo.reshape(len(x_train_crudo), -1)
        x_test_a_norm = x_test_crudo.reshape(len(x_test_crudo), -1)

    x_train_norm = normalizador.fit_transform(x_train_a_norm)
    x_test_norm = normalizador.transform(x_test_a_norm)
    
    cargador.guardar_conjunto_procesado(
        x_train_norm, x_test_norm, y_train_crudo, y_test_crudo,
        dir_salida='../data/processed/',
        nombre_normalizador=nombre,
        meta={'fuente_normalizacion': 'master_notebook.ipynb'}
    )

## 6. Decisión Operativa: Candidatos a Dimensión Reducida

In [ ]:
n_mi = sum(serie_mi > 0.01)

dims_candidatas = sorted(list(set([
    int(np.ceil(estimaciones_dim['TwoNN'])),
    umbrales_pca['n_componentes_90_varianza'],
    umbrales_pca['n_componentes_95_varianza'],
    n_mi,
    10, 20, 50, 100
])))

df_dims_candidatas = pd.DataFrame(dims_candidatas, columns=['dimension_candidata'])
df_dims_candidatas.to_csv('../outputs/tables/dimensiones_candidatas.csv', index=False)

print("Dimensiones candidatas para la fase de reducción:")
print(dims_candidatas)
print("\nGuardado en: ../outputs/tables/dimensiones_candidatas.csv")

**--- Fin del Bloque A ---**